In [2]:
from nimber_ops import *
from transfinite import w, Ordinal

In [3]:
Ordinal(w, 2, 3)

w**w*2 + 3

In [10]:
class Nim:
    ''' nimbers '''
    def __init__(self, n : int | Ordinal) -> None:
        ''' ordinal considered an a field element in On_2 
        val = ordinal
        field = smallest x > n such that x is a field
        base = largest y < n such that y is a field (or base = 0 for n < 2)
        write n = high * base + low, where low,high < base
        '''
        if isinstance(n, int):
            self.val = abs(n)
            self.isfinite = True            
            if self.val < 2:
                self.field = 2 # smallest 
                self.base = 0
                self.high = 0
                self.low = self.val
            else:
                level = 0
                while n >> (1 << level) > 0:
                    level += 1
                self.field = exp(exp(level))
                self.base = exp(exp(level - 1))
                self.high = self.val // self.base
                self.low = self.val - self.high * self.base
        else:
            assert isinstance(n, Ordinal), 'An infinite nimber must be an ordinal'
            self.val = n
            self.isfinite = False
            self.field = ... # to do
            self.base = ...
            self.high = ...
            self.low = ...
    
    def _repr_latex_(self):
        """
        Special method for Jupyter to render LaTeX.
        """
        if self.isfinite:
            # No special LaTeX for integers, just return the string
            return f"${self.val}$"
        else:
            # Delegate to the Ordinal's LaTeX representation
            return self.val._repr_latex_()
            
    def __repr__(self) -> str:
        if self.isfinite:       
            return str(self.val)
        else:
            return self.val.__repr__()
    
    def __add__(self, other):
        if self.isfinite & other.isfinite:
            return Nim(self.val ^ other.val)
    
    def __mul__(self, other):
        x, y = self.val, other.val
        if self.isfinite and other.isfinite:
            def nim_product(a : int, b : int) -> int:
                # first handle trivial cases
                if a == 0 or b == 0:
                    return 0
                elif a == 1:
                    return b
                elif b == 1:
                    return a
                elif a == 2 and b == 2:
                    return 3
                else:
                    # do euclidean division by greatest possible fermat power 
                    # a = q_a * F_a + r_a and b = q_b * F_b + r_b
                    F_a, q_a, r_a = Nim(a).base, Nim(a).high, Nim(a).low
                    F_b, q_b, r_b = Nim(b).base, Nim(b).high, Nim(b).low
                    
                    # if one the Fermat powers is greater than the other, then
                    # nim multiplication by it is the same as ordinary multiplication
                    if F_a < F_b:
                        return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
                    elif F_a > F_b:
                        return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
                    else:
                        # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
                        p_1 = nim_product(q_a,q_b)
                        p_2 = nim_product(r_a,r_b)
                        p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
                        p_4 = nim_product(p_1, F_a >> 1)
                        p_5 = p_3 ^ p_2
                        return p_5 * F_a ^ p_2 ^ p_4
            return Nim(nim_product(x, y))  
        
    def sqrt(self):
        if self.isfinite:
            if self.field == 2:
                return self
            term = self**2 + self
            return term.sqrt() + self
        
    def inv(self):
        return self ** (-1)
        
    def __pow__(self, p):
        """
        Compute x**n using exponentiation by squaring.

        """
        if self.isfinite:
            if p >= 0: # binary exponentiation by squaring
                result = Nim(1) 
                while p > 0:
                    if p & 1:
                        result = result * self
                    self = self * self
                    p >>= 1
                return result
            elif p == -1:
                if self.field == 2: return self
                a, b, F, f = Nim(self.high), Nim(self.low), Nim(self.base), Nim(self.base >> 1)
                det = (a + b)*b + a*a*f
                return det**(-1) * (a*F + (a+b))
            else:
                inv = self ** (-1)
                return inv ** (-p)    

In [13]:
a = Nim(1923847548392) ** (-12)
b = Nim(1923847548392) ** (12)
a, b, a*b

(3322229293480365377, 8254917243207255880, 1)

In [ ]:
# class Nim:
#     ''' finite nimbers '''
#     def __init__(self, n : int) -> None:
#         self.ord = abs(n)
#         # n = high * 2^(2 ^ (level-1)) + low;   high, low < 2^(2 ^ (level-1))
#         level = 0
#         if self.ord < 2:
#             self.level = level
#             self.fermat = None
#             self.high = 0
#             self.low = self.ord
#         else:
#             while n >> (1 << level) > 0:
#                 level += 1
#             self.level = level
#             self.fermat = (1 << (1 << (level - 1)))
#             self.high = self.ord // self.fermat
#             self.low = self.ord - self.high * self.fermat 
#     def __repr__(self) -> str:       
#         return str(self.ord)
    
#     def __add__(self, other):
#         return Nim(self.ord ^ other.ord)
    
#     def __mul__(self, other):
#         x = self.ord
#         y = other.ord
        
#         def nim_product(a : int, b : int) -> int:
#             # first handle trivial cases
#             if a == 0 or b == 0:
#                 return 0
#             elif a == 1:
#                 return b
#             elif b == 1:
#                 return a
#             elif a == 2 and b == 2:
#                 return 3
#             else:
#                 # do euclidean division by greatest possible fermat power 
#                 # a = q_a * F_a + r_a and b = q_b * F_b + r_b
#                 F_a, q_a, r_a = Nim(a).fermat, Nim(a).high, Nim(a).low
#                 F_b, q_b, r_b = Nim(b).fermat, Nim(b).high, Nim(b).low
                
#                 # if one the Fermat powers is greater than the other, then
#                 # nim multiplication by it is the same as ordinary multiplication
#                 if F_a < F_b:
#                     return nim_product(a,q_b)*F_b ^ nim_product(a,r_b)
#                 elif F_a > F_b:
#                     return nim_product(q_a,b)*F_a ^ nim_product(r_a,b)
#                 else:
#                     # otherwise we have to distribute and use F_n ** 2 = 3 * F_n / 2
#                     p_1 = nim_product(q_a,q_b)
#                     p_2 = nim_product(r_a,r_b)
#                     p_3 = nim_product(q_a ^ r_a, q_b ^ r_b)
#                     p_4 = nim_product(p_1, F_a >> 1)
#                     p_5 = p_3 ^ p_2
#                     return p_5 * F_a ^ p_2 ^ p_4
#         return Nim(nim_product(x, y))